# PersonaPlex — RunPod RTX 5090 Deployment Notebook

This notebook deploys **[PersonaPlex](https://github.com/NVIDIA/personaplex)** — NVIDIA's real-time, full-duplex
speech-to-speech model with persona/voice control (built on [Moshi](https://arxiv.org/abs/2410.00037)) — on a
**fresh RunPod pod with an RTX 5090 GPU**.

It is derived directly from the repository's own `README.md`, `client/README.md`, `moshi/pyproject.toml`,
`moshi/requirements.txt`, and the server/offline entrypoints (`moshi/moshi/server.py`, `moshi/moshi/offline.py`,
`moshi/moshi/utils/connection.py`, `moshi/moshi/models/loaders.py`). No steps are invented — every command below
maps to something the repo actually does or documents.

## What this notebook does

1. Verifies the RunPod environment, GPU and CUDA.
2. Sets up persistent storage on the RunPod volume.
3. Installs system + Python dependencies (including the **Blackwell/RTX 5090‑specific PyTorch build**).
4. Clones the repository (skipped automatically if you already uploaded it).
5. Configures Hugging Face authentication and downloads the model weights, tokenizer, voices and web UI assets.
6. Starts the PersonaPlex backend server (which also serves the prebuilt web UI — no separate frontend build is
   required for normal use).
7. Verifies the deployment with an automated offline inference smoke test and an HTTP check.
8. Documents GPU optimization notes and troubleshooting steps.

## What PersonaPlex does *not* have (so these checklist items are intentionally empty)

- **No database** of any kind — there is nothing to initialize.
- **No separate "API server"** — the single aiohttp server in `moshi/moshi/server.py` exposes both the WebSocket
  endpoint (`/api/chat`) and the static web UI on one port.
- **No Gradio/Streamlit app** — the only Gradio dependency is `gradio.networking.setup_tunnel`, an *optional*
  public-URL tunnel (`--gradio-tunnel`), not a Gradio UI. RunPod's own HTTP port proxy makes this unnecessary here.
- **No required "configuration file" to generate** — runtime behavior is controlled entirely by CLI flags and the
  HF-hosted `config.json` that ships with the model weights.

## Things this notebook *cannot* automate (you must do these yourself)

1. **Accept the NVIDIA Open Model License** for the gated model repo
   [`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1) — log into Hugging Face in a
   browser and click "Agree and access repository". No script can click this for you.
2. **Create a Hugging Face access token** for that same account and have it ready to paste into the auth cell
   below.
3. **Expose the server port (default `8998`) as an HTTP Service** from the RunPod pod's *Connect* page (RunPod
   console action) so the proxy URL works from outside the pod.
4. **Grant microphone permission** in your browser when you open the live web UI — this is a per-user browser
   security prompt.

Run the cells **top to bottom**. The only cell you must intentionally run out of the normal flow is the final
**"Stop the server"** cleanup cell — leave it for whenever you actually want to shut the server down.

## 1. Environment sanity checks

Confirms this is a Linux pod with a GPU attached and a supported Python version (`moshi-personaplex` requires
Python ≥ 3.10, per `moshi/pyproject.toml`).

In [ ]:
import platform
import sys

print("Platform:", platform.platform())
print("Python:", sys.version)

assert sys.version_info >= (3, 10), (
    f"PersonaPlex (moshi/pyproject.toml) requires Python >= 3.10, found {sys.version_info}."
)
print("Python version OK.")

In [ ]:
# Confirm the NVIDIA driver sees a GPU at the OS level before we install anything.
!nvidia-smi

## 2. Persistent storage setup (RunPod volume)

RunPod mounts your persistent Network Volume at **`/workspace`**. Anything written there survives pod
stop/start (model weights are multiple GB, so re-downloading them on every restart would be wasteful).
If `/workspace` isn't present (e.g. you're running this notebook somewhere else), we fall back to the home
directory so the notebook still works end-to-end.

We also point Hugging Face's cache (`HF_HOME`) at the persistent volume so `huggingface_hub` downloads
(triggered both by this notebook and internally by `moshi/moshi/server.py` / `moshi/moshi/offline.py`) are
cached once and reused across restarts.

In [ ]:
import os

WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_URL = "https://github.com/MoshiHead/personaplex-original-code-streaming-prompt-long-conversation.git"
REPO_DIR = os.path.join(WORKSPACE, "personaplex")
HF_CACHE_DIR = os.path.join(WORKSPACE, ".cache", "huggingface")
HF_REPO_ID = "nvidia/personaplex-7b-v1"   # loaders.DEFAULT_REPO in moshi/moshi/models/loaders.py

SERVER_HOST = "0.0.0.0"   # must be 0.0.0.0 (not "localhost") so RunPod's proxy can reach the server
SERVER_PORT = 8998        # default port used by moshi.server

# RunPod's HTTP port proxy terminates TLS at its edge and forwards plain HTTP to the container, so by
# default we do NOT enable the app's own self-signed TLS (--ssl). Flip this to True only if you plan to
# expose SERVER_PORT directly as a raw TCP port instead of through RunPod's HTTP proxy.
USE_APP_TLS = False

# An RTX 5090 has 32GB of VRAM, comfortably enough for this 7B model in bf16 -- CPU offload should not be
# needed. Flip to True only if you hit CUDA OOM (requires the `accelerate` package, installed below anyway).
USE_CPU_OFFLOAD = False

os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
# Keep PATH aware of ~/.local/bin, where moshi/moshi/utils/connection.py installs `mkcert` if --ssl is used.
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + os.pathsep + os.environ.get("PATH", "")

print("WORKSPACE   :", WORKSPACE)
print("REPO_DIR    :", REPO_DIR)
print("HF_HOME     :", os.environ["HF_HOME"])
print("USE_APP_TLS :", USE_APP_TLS)

## 3. System package installation

Per the repo's `README.md` prerequisites: install the **Opus codec development library** before installing the
Python package (it's required by `sphn`, which PersonaPlex uses for Opus audio streaming over the WebSocket).
We also make sure `git` is present for cloning.

In [ ]:
import os

SUDO = "" if os.geteuid() == 0 else "sudo "

!{SUDO}apt-get update -qq
!{SUDO}apt-get install -y -qq --no-install-recommends git ca-certificates libopus-dev
print("System packages installed.")

## 4. Repository cloning

If `REPO_DIR` doesn't already contain the project (e.g. you uploaded your own copy of this repo to the volume
beforehand), it is cloned fresh from GitHub. If it's already there, cloning is skipped automatically — this
cell is safe to re-run on every pod restart.

In [ ]:
import pathlib
import subprocess

repo_marker = pathlib.Path(REPO_DIR) / "moshi" / "pyproject.toml"

if repo_marker.exists():
    print(f"Repository already present at {REPO_DIR}, skipping clone.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}.")

assert repo_marker.exists(), f"Expected {repo_marker} to exist after cloning/upload."


## 5. Python dependency installation

The repo's documented install command is:

```bash
pip install moshi/.
```

which installs from `moshi/pyproject.toml` (`numpy`, `safetensors`, `huggingface-hub`, `einops`,
`sentencepiece`, `sounddevice`, `sphn`, `torch>=2.2,<2.5`, `aiohttp`).

### RTX 5090 / Blackwell note

The pinned `torch<2.5` build does **not** ship CUDA kernels for Blackwell (RTX 50‑series, `sm_120`) GPUs. The
repo's `README.md` explicitly documents the fix for this
([NVIDIA/personaplex#2](https://github.com/NVIDIA/personaplex/issues/2)): reinstall PyTorch from the `cu130`
wheel index *after* the base install. This intentionally overrides the `<2.5` pin — that's expected and is the
upstream-recommended fix, not a mistake.

In [ ]:
%pip install -q --upgrade pip setuptools wheel
%pip install -q "{REPO_DIR}/moshi/." 

In [ ]:
# Blackwell (RTX 5090) requires CUDA-13.0-built PyTorch wheels. This intentionally supersedes the
# torch<2.5 pin from moshi/pyproject.toml -- see README.md "Extra step for Blackwell based GPUs".
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

In [ ]:
# `accelerate` enables --cpu-offload (README's "CPU Offload" section); `gradio` enables the optional
# --gradio-tunnel fallback for public exposure if you ever need it instead of RunPod's HTTP proxy.
%pip install -q accelerate gradio

## 6. CUDA / GPU verification

Confirms PyTorch can see the RTX 5090 and that the installed build actually has working CUDA kernels for it
(a bf16 matmul smoke test) — this is the check that would have failed before the `cu130` reinstall above if it
had been skipped.

In [ ]:
import torch

print("Torch version      :", torch.__version__)
print("Torch CUDA version :", torch.version.cuda)
print("CUDA available     :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected by PyTorch. Confirm this RunPod pod has an RTX 5090 GPU attached "
        "and that the driver is healthy (see the `nvidia-smi` output above)."
    )

device_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
print("GPU                :", device_name)
print("Compute capability :", capability)

if "5090" not in device_name and "RTX 50" not in device_name:
    print(f"WARNING: expected an RTX 5090, found '{device_name}'. Continuing anyway.")

# Smoke test: this is exactly the kind of op that fails with
# "no kernel image is available for execution on the device" on Blackwell if torch wasn't
# reinstalled from the cu130 index.
x = torch.randn(4096, 4096, device="cuda", dtype=torch.bfloat16)
y = x @ x
torch.cuda.synchronize()
print("bf16 CUDA matmul smoke test OK, result shape:", tuple(y.shape))

## 7. Hugging Face authentication

**Manual step required before running this cell:** log into Hugging Face in a browser, open
[`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1), and click **"Agree and access
repository"** to accept the NVIDIA Open Model License. Then create an access token (read access is enough) at
<https://huggingface.co/settings/tokens>.

The repo's `README.md` documents this as `export HF_TOKEN=<YOUR_HUGGINGFACE_TOKEN>`; the cell below does the
same thing from inside the notebook, via a hidden prompt so the token isn't echoed into cell output.

In [ ]:
from getpass import getpass

from huggingface_hub import login

# hf_token = os.environ.get("HF_TOKEN")
# sa token...not mine
hf_token = 'tLNSyNjFduNaLUbvyxosVqiGwuAtiQPOTt'    
if not hf_token:
    hf_token = getpass("Enter your Hugging Face access token (input hidden): ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
print("Logged in to Hugging Face Hub.")

## 8. Model downloading

`moshi/moshi/server.py` and `moshi/moshi/offline.py` both lazily call `hf_hub_download` for each asset the
first time they need it. We pre-fetch the same files here so that (a) any license/token problem surfaces now
with a clear error instead of mid-startup, and (b) everything is already warm in the `HF_HOME` cache before we
launch the server.

Assets (from `moshi/moshi/models/loaders.py` and `moshi/moshi/server.py`):
- `config.json` — model config
- `tokenizer_spm_32k_3.model` — SentencePiece text tokenizer
- `tokenizer-e351c8d8-checkpoint125.safetensors` — Mimi codec weights
- `model.safetensors` — Moshi/PersonaPlex LM weights
- `voices.tgz` — packaged voice-prompt embeddings (NATF0‑3, NATM0‑3, VARF0‑4, VARM0‑4)
- `dist.tgz` — the **prebuilt web UI** (this is why no separate frontend build step is needed)

In [ ]:
import tarfile

from huggingface_hub import hf_hub_download

ASSET_FILES = [
    "config.json",
    "tokenizer_spm_32k_3.model",
    "tokenizer-e351c8d8-checkpoint125.safetensors",
    "model.safetensors",
    "voices.tgz",
    "dist.tgz",
]

downloaded = {}
try:
    for fname in ASSET_FILES:
        # No explicit cache_dir: this honors HF_HOME (set in step 2) so the notebook and the server
        # subprocess we launch later (which inherits the same env) share one cache.
        path = hf_hub_download(HF_REPO_ID, fname)
        downloaded[fname] = path
        print(f"OK  {fname} -> {path}")
except Exception as e:
    raise RuntimeError(
        "Failed to download model assets from "
        f"https://huggingface.co/{HF_REPO_ID}. This almost always means either:\n"
        "  1) you have not clicked 'Agree and access repository' on that model page yet, or\n"
        "  2) the HF_TOKEN you supplied doesn't belong to the account that accepted the license, or\n"
        "  3) the token is invalid/expired.\n"
        f"Original error: {e}"
    )

In [ ]:
import pathlib

# Pre-extract the tarballs once, exactly like _get_voice_prompt_dir / _get_static_path do in
# moshi/moshi/server.py, so the first real request doesn't pay the extraction cost.
for tgz_name in ("voices.tgz", "dist.tgz"):
    tgz_path = pathlib.Path(downloaded[tgz_name])
    out_dir = tgz_path.parent / tgz_name.replace(".tgz", "")
    if not out_dir.exists():
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(path=tgz_path.parent)
    print(f"{tgz_name} -> {out_dir} ({'already extracted' if out_dir.exists() else 'extracted now'})")

## 9. TLS certificate directory (only used if `USE_APP_TLS = True`)

The README's default local-machine workflow is:

```bash
SSL_DIR=$(mktemp -d); python -m moshi.server --ssl "$SSL_DIR"
```

`--ssl` makes `moshi/moshi/utils/connection.py` auto-install `mkcert` and generate a **self-signed**
certificate in that directory. That's appropriate for direct LAN/local access, but on RunPod we're putting the
server behind RunPod's own HTTPS proxy (Section 11), which already terminates TLS at the edge — adding a second,
self-signed TLS layer underneath it would just produce certificate warnings for no benefit. We still create the
directory here so you can flip `USE_APP_TLS = True` above if you'd rather expose `SERVER_PORT` as a raw TCP port
instead of through the HTTP proxy.

In [ ]:
import tempfile

SSL_DIR = tempfile.mkdtemp(prefix="personaplex_ssl_")
print("SSL_DIR:", SSL_DIR, "(only used if USE_APP_TLS = True)")

## 9b. Build the web client from source — recommended for long conversations, required for a longer Text Prompt

By default `python -m moshi.server` does **not** use the `client/` source in this repo at all: when
`--static` is omitted, `_get_static_path` in `moshi/moshi/server.py` downloads NVIDIA's **prebuilt** web UI
bundle (`dist.tgz`, extracted in Section 8) and serves that instead.

**This section is no longer purely cosmetic.** This checkout's `client/src/pages/Conversation/hooks/useSocket.ts`
and `Conversation.tsx` contain the client-side reliability fixes for long conversations (automatic WebSocket
reconnection, saner inactivity timeout — see Section 15b). NVIDIA's prebuilt `dist.tgz` bundle predates those
fixes and does **not** contain them. If you skip this section, the server falls back to that prebuilt bundle
and you are running the *old*, unpatched client — those fixes (and the raised Text Prompt limit below) will
not actually be in effect even though the server-side fixes in `server.py` are.

Run the two cells below to:
1. Install Node.js, matching `client/.nvmrc`.
2. Patch the cloned repo's `Queue.tsx` to raise the Text Prompt limit to `TEXT_PROMPT_MAX_LEN` characters and
   add a "Topic-only (grounded)" preset.
3. Build the client (`npm ci && npm run build`, per `client/README.md`) and record the build output so
   Section 10 can launch the server with `--static` pointing at it instead of the prebuilt bundle.

If the build fails for any reason, the notebook falls back to NVIDIA's prebuilt UI automatically (with a
1000-char Text Prompt limit and without the reconnect fix) so the rest of the flow still completes — but check
the printed error and retry rather than relying on that fallback for a long-conversation deployment.

In [ ]:
import shutil
import subprocess

NODE_MAJOR = "20"  # matches client/.nvmrc (v20.12.2)

def _node_major_ok():
    node = shutil.which("node")
    if not node:
        return False
    out = subprocess.run([node, "-v"], capture_output=True, text=True).stdout.strip()
    return out.startswith(f"v{NODE_MAJOR}.")

if _node_major_ok():
    print("Node.js already installed:", subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip())
else:
    !{SUDO}apt-get install -y -qq curl
    !curl -fsSL https://deb.nodesource.com/setup_{NODE_MAJOR}.x | {SUDO}bash -
    !{SUDO}apt-get install -y -qq nodejs
    print("Node.js installed:", subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip())


In [ ]:
import os
import pathlib
import subprocess

TEXT_PROMPT_MAX_LEN = 4000  # raise/lower as needed — see the context-window note in Section 13b below

queue_tsx = pathlib.Path(REPO_DIR) / "client" / "src" / "pages" / "Queue" / "Queue.tsx"
src = queue_tsx.read_text(encoding="utf-8")
patched = src.replace("maxLength={1000}", f"maxLength={{{TEXT_PROMPT_MAX_LEN}}}")
patched = patched.replace("{textPrompt.length}/1000", f"{{textPrompt.length}}/{TEXT_PROMPT_MAX_LEN}")
if patched != src:
    queue_tsx.write_text(patched, encoding="utf-8")
    print(f"Patched {queue_tsx} -> Text Prompt limit is now {TEXT_PROMPT_MAX_LEN} characters.")
else:
    print(f"{queue_tsx}: pattern not found (already patched, or upstream file changed) — left unchanged.")

STATIC_DIR = None
client_dir = os.path.join(REPO_DIR, "client")
try:
    subprocess.run(["npm", "ci"], cwd=client_dir, check=True)
    subprocess.run(["npm", "run", "build"], cwd=client_dir, check=True)
    built_dist = os.path.join(client_dir, "dist")
    assert os.path.isdir(built_dist), f"Expected {built_dist} to exist after `npm run build`."
    STATIC_DIR = built_dist
    print("Custom client built at:", STATIC_DIR)
except Exception as e:
    print(f"Client build failed ({e!r}); falling back to NVIDIA's prebuilt UI (1000-char Text Prompt limit).")
    STATIC_DIR = None


## 10. Backend (+ web UI) startup

This is the repo's **recommended and only documented launch method** — `python -m moshi.server`. It is a
single aiohttp process that serves:
- the WebSocket endpoint `/api/chat` (the real-time speech protocol), and
- the web UI at `/` — either the prebuilt `dist.tgz` bundle (downloaded in Section 8), or your own
  client build from Section 9b if `STATIC_DIR` was set there (passed to the server via `--static`).
  Either way there is **no separate frontend process to start** for normal use.

A notebook cell that calls `web.run_app(...)` directly would block the kernel forever, so we launch it as a
background subprocess and log to a file, then poll that log in the next cell.

In [ ]:
import subprocess
import sys

env = os.environ.copy()
LOG_PATH = os.path.join(WORKSPACE, "personaplex_server.log")

# Mute recovery (see Section 15b): if the model produces no text for this many seconds
# mid-conversation, the server instantly restores a snapshot of its state from right after the
# voice/text prompt finished loading. Set to 0 to disable.
#
# For a CLEAN EVIDENCE-GATHERING run of the ~6:05 silence issue (see Section 15c), leave this at
# 0 -- recovery would auto-fix the natural failure before its full signature is captured. Set it
# back to a nonzero value (e.g. 10-30) for normal day-to-day use once you're done investigating.
MUTE_RECOVERY_SECS = 0

# Forensic diagnostic instrumentation (see Section 15c). DIAG_INTERVAL_SECS controls how often the
# structured JSONL log (personaplex_diag.jsonl) is written; DIAG_PROBS additionally captures the
# model's own token-sampling probabilities every step (small added overhead, off by default -- turn
# on for one especially detailed run if the basic instrumentation doesn't already answer the
# question). DIAG_DIR is where personaplex_diag.jsonl and all snapshot_*.json files are written.
DIAG_INTERVAL_SECS = 5.0
DIAG_PROBS = False
DIAG_DIR = WORKSPACE

cmd = [sys.executable, "-m", "moshi.server", "--host", SERVER_HOST, "--port", str(SERVER_PORT)]
if USE_APP_TLS:
    cmd += ["--ssl", SSL_DIR]
if USE_CPU_OFFLOAD:
    cmd += ["--cpu-offload"]
if globals().get("STATIC_DIR"):
    cmd += ["--static", STATIC_DIR]
if MUTE_RECOVERY_SECS > 0:
    cmd += ["--mute-recovery-secs", str(MUTE_RECOVERY_SECS)]
cmd += ["--diag-interval-secs", str(DIAG_INTERVAL_SECS), "--diag-dir", DIAG_DIR]
if DIAG_PROBS:
    cmd += ["--diag-probs"]

print("Launching:", " ".join(cmd))

log_file = open(LOG_PATH, "w")
server_proc = subprocess.Popen(
    cmd, cwd=os.path.join(REPO_DIR, "moshi"), env=env, stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"Server launched with PID {server_proc.pid}. Logs: {LOG_PATH}")
print(f"Diagnostic JSONL + snapshots will be written under: {DIAG_DIR}")

In [ ]:
import time

def tail(path, n=60):
    with open(path) as f:
        return "".join(f.readlines()[-n:])

READY_MARKER = "Access the Web UI directly at"
TIMEOUT_S = 900  # first run downloads + loads a multi-GB model; be generous
POLL_S = 5

start = time.time()
ready = False
while time.time() - start < TIMEOUT_S:
    if server_proc.poll() is not None:
        print(tail(LOG_PATH))
        raise RuntimeError(
            f"Server process exited early with return code {server_proc.returncode}. See log above."
        )
    if READY_MARKER in open(LOG_PATH).read():
        ready = True
        break
    time.sleep(POLL_S)

print(tail(LOG_PATH))

if not ready:
    raise TimeoutError(
        f"Server did not report readiness within {TIMEOUT_S}s. Check the log tail above -- "
        "the most common causes are a slow first-time model download or a CUDA/driver mismatch."
    )

print("\nServer is up and warmed up.")

## 11. Expose the port & get the access URL

RunPod will not route external traffic to a port unless you explicitly expose it. In the pod's **Connect**
page, add an **HTTP Service** for `SERVER_PORT` (default `8998`) if you haven't already. RunPod then serves it
at `https://<POD_ID>-<PORT>.proxy.runpod.net`, with RunPod terminating TLS — which is exactly why
`USE_APP_TLS = False` is the right default here (see Section 9).

In [ ]:
pod_id = os.environ.get("RUNPOD_POD_ID")

if pod_id:
    public_url = f"https://{pod_id}-{SERVER_PORT}.proxy.runpod.net"
    print("RunPod public URL (requires SERVER_PORT to be exposed as an HTTP Service on the pod's Connect page):")
    print(" ", public_url)
else:
    print(
        "RUNPOD_POD_ID was not found in the environment. If you are on RunPod, expose "
        f"port {SERVER_PORT} as an HTTP Service from the pod's Connect page and use the proxy URL shown there."
    )

scheme = "https" if USE_APP_TLS else "http"
print(f"Local URL inside the pod: {scheme}://localhost:{SERVER_PORT}")

## 12. Verification — offline inference smoke test

This runs the repo's documented `moshi.offline` "Assistant example" exactly as given in `README.md` — it
streams `assets/test/input_assistant.wav` through the model with voice prompt `NATF2.pt` and writes an output
WAV + JSON transcript. This doesn't need a browser, microphone, or open port, so it's a good way to confirm the
backend, weights and GPU are all working correctly before testing the live UI.

In [ ]:
import subprocess
import sys

offline_cmd = [
    sys.executable, "-m", "moshi.offline",
    "--voice-prompt", "NATF2.pt",
    "--input-wav", "assets/test/input_assistant.wav",
    "--seed", "42424242",
    "--output-wav", "output_assistant.wav",
    "--output-text", "output_assistant.json",
]

result = subprocess.run(
    offline_cmd, cwd=REPO_DIR, env=os.environ.copy(), capture_output=True, text=True,
)

print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise RuntimeError("Offline smoke test failed -- see output above.")

print("\nOffline smoke test succeeded.")

In [ ]:
import json

from IPython.display import Audio, display

output_wav_path = os.path.join(REPO_DIR, "output_assistant.wav")
output_json_path = os.path.join(REPO_DIR, "output_assistant.json")

display(Audio(output_wav_path))

with open(output_json_path) as f:
    tokens = json.load(f)
print("Generated text tokens (joined):")
print("".join(tokens))

## 13. Verification — HTTP check on the live server

Confirms the running server process answers on `/` (the web UI's `index.html`, served from `dist.tgz`).

In [ ]:
import ssl
import urllib.request

scheme = "https" if USE_APP_TLS else "http"
ctx = ssl._create_unverified_context() if USE_APP_TLS else None

with urllib.request.urlopen(f"{scheme}://localhost:{SERVER_PORT}/", context=ctx, timeout=15) as resp:
    print("HTTP status :", resp.status)
    print("Content-Type:", resp.headers.get("Content-Type"))
    assert resp.status == 200, f"Expected 200, got {resp.status}"

print("Web UI is being served correctly.")

## 13b. Provide topic information for grounded responses

PersonaPlex exposes exactly one channel for "give it information, then make it stick only to that": the
**Text Prompt** field already in the web UI. That field *is* the model's system prompt — `server.py` wraps
whatever you type in `<system> ... <system>` tags (`wrap_with_system_tags`) and feeds it to the model token
by token as the very first thing in its context, before either of you say a word
(`moshi/moshi/models/lm.py: step_system_prompts`). There is no separate "context" field or knowledge base —
everything goes through this one box.

**Where to put it:** type/paste it into the Text Prompt box in Section 14 below, *before* you click connect.
Anything you only say out loud after connecting is ordinary spoken conversation, not a system prompt — it
is not privileged the way text typed into that box is.

**How to phrase it:** PersonaPlex's own training data already has a pattern for exactly this — a short
role sentence followed by `Information: <facts>` (see the "Customer Service Roles" examples in
`README.md`). That pattern is in-distribution for the model, so it follows it more reliably than a generic
"only answer from this text" instruction wrapped around a long free-form document. The **"Topic-only
(grounded)"** preset added to the web UI (`client/src/pages/Queue/Queue.tsx`, only present if you built the
client in Section 9b) gives you a starting template — replace its bracketed placeholders with your topic
and facts. The cell below builds and prints the same kind of string for you to copy in.

**Limits this does not remove:**
1. This is a soft instruction a 7B model tries to follow, not a hard filter. There is no
   retrieval/grounding-enforcement layer in this codebase, so it can still drift, guess, or blend in outside
   knowledge — "strict" here is best-effort prompting, not a guarantee.
2. The model's causal attention only spans the last `context = 3000` steps
   (`moshi/moshi/models/loaders.py`) at `FRAME_RATE = 12.5` Hz — about 4 minutes of combined
   prompt-loading + silence + conversation. Every character of your Text Prompt is loaded as its own step
   before the conversation starts and counts against that same 4-minute budget: a longer prompt means a
   longer pause before you can start talking, *and* less of the window left before the model can no longer
   attend back to your original prompt at all (it falls out of the attention window like anything else
   `context` steps in the past). There is no mechanism in this code to re-inject the prompt mid-conversation
   — for sessions longer than a few minutes, treat the grounding as something that fades, not something
   permanent.
3. Given (2), prefer a tight list of concrete facts (closer to the README's examples) over a long prose
   document, even though Section 9b raises the box's limit up to `TEXT_PROMPT_MAX_LEN` characters.

In [ ]:
CUSTOM_TOPIC_INFO = (
    "Replace this with your facts, e.g.: Our return policy allows returns within 30 days with a receipt. "
    "Store hours are 9am-6pm Monday-Saturday, closed Sunday."
)
ROLE_SENTENCE = "You are a knowledgeable assistant for [your topic]."

grounded_prompt = (
    f"{ROLE_SENTENCE} Only use the Information below to answer questions; if something is not covered by "
    f"it, say it is outside what you were told and you cannot help with that. Do not use any other "
    f"knowledge. Information: {CUSTOM_TOPIC_INFO}"
)

limit = globals().get("TEXT_PROMPT_MAX_LEN", 1000)
print(f"Characters: {len(grounded_prompt)} (Text Prompt box limit: {limit})")
print()
print(grounded_prompt)
print()
print("Copy the text above into the 'Text Prompt' box in Section 14, before clicking connect.")


## 14. Using the live web UI

Open the URL printed in Section 11 in a browser, allow microphone access when prompted (this is the one
browser permission step that can't be automated), and start talking.

### Voices (from `README.md`)

```
Natural(female): NATF0, NATF1, NATF2, NATF3
Natural(male):   NATM0, NATM1, NATM2, NATM3
Variety(female): VARF0, VARF1, VARF2, VARF3, VARF4
Variety(male):   VARM0, VARM1, VARM2, VARM3, VARM4
```

### Example role prompts (from `README.md`)

- Assistant role: `You are a wise and friendly teacher. Answer questions or provide advice in a clear and
  engaging way.`
- Casual conversation: `You enjoy having a good conversation.`
- Customer service example: `You work for CitySan Services which is a waste management company and your name
  is Ayelen Lucero. Information: Verify customer name Omar Torres. Current schedule: every other week.
  Upcoming pickup: April 12th. Compost bin service available for $8/month add-on.`

See the repo's `README.md` "Prompting Guide" section for more examples and guidance.

## 15. GPU optimization notes for RTX 5090

- **bf16 by default**: `moshi/moshi/models/loaders.py:get_moshi_lm` loads the LM in `torch.bfloat16`. Blackwell
  has fast native bf16 Tensor Core throughput, so no dtype changes are needed.
- **`--cpu-offload` is unnecessary here**: it exists in `server.py`/`offline.py` for GPUs with insufficient
  VRAM (via `accelerate`'s `infer_auto_device_map`). An RTX 5090's 32GB comfortably fits this 7B model plus the
  Mimi codec; only enable `USE_CPU_OFFLOAD` if you're also running other large workloads on the same GPU.
- **One process = one GPU's worth of model**: `ServerState.__init__` loads two Mimi instances and the LM once
  per process and keeps them resident (`streaming_forever`). Don't launch a second `moshi.server` process
  against the same GPU unless you've confirmed there's VRAM headroom — check with `nvidia-smi`.
- **Warmup cost is already handled**: `state.warmup()` runs 4 dummy frames through the full encode → LM →
  decode path right after model load, ahead of any real connections — that's the per-process latency you saw
  the log wait for in Section 10, not something to optimize further.
- **CUDA build must match the GPU**: Blackwell (`sm_120`) needs the `cu130` PyTorch wheels installed in
  Section 5. If you ever see `no kernel image is available for execution on the device`, that cell needs to be
  re-run (something likely reinstalled a different torch build afterward).

## 15b. Long-conversation reliability ("goes silent after ~5-6 minutes")

**Symptom:** a live conversation works normally for the first several minutes, then the model
stops replying -- no more speech, no new transcript text -- while the connection stays green.

**What the diagnostics proved (evidence, not assumption):** instrumentation added to
`server.py` (the `perf:` log line every 5s + a hang watchdog that stack-dumps all threads if any
pipeline sub-step stalls > 8s) showed that during the "freeze" the pipeline is **completely
healthy**: mic audio keeps arriving, the model keeps stepping at a steady ~0.45-0.5x real-time
ratio, output audio keeps being generated and sent, the input backlog stays at ~0s, and the
watchdog never fires. Nothing hangs, nothing leaks, nothing times out. **The model itself goes
content-silent**: it keeps generating tokens, but only PAD/silence -- so the user hears nothing
and no text appears, indefinitely.

**Why:** the model's causal attention is a fixed rotating window of `context = 3000` steps at
12.5 steps/s -- **4 minutes, total**, shared by the voice prompt, the text prompt (one step per
token!), and the conversation itself. A ~4,000-character text prompt consumes ~1,000+ of those
steps before anyone speaks. Once the session outlives the window, the persona/system prompt has
completely rotated out of what the model can attend to, and generation degenerates into
permanent silence. The ~5-6-minute mark is simply (window - prompt length) + however long the
model coasts before fully degrading. This is inherent to the released checkpoint (the window
size is baked into its trained position range and the `RingKVCache` capacity), not a bug in the
serving code.

**Mitigations in this checkout:**

1. **Instant server-side mute recovery (`--mute-recovery-secs N`, wired up in Section 10 as
   `MUTE_RECOVERY_SECS`, default 10):** if the model produces no text for N seconds
   mid-conversation, the server jumps the model's entire streaming state (every attention layer's
   rotating KV-cache, LMGen's own token cache/offset) back to a snapshot taken right after the
   voice+text prompt finished loading at connection time. This is a handful of in-place tensor
   `.copy_()` calls -- no forward passes, no token-by-token replay -- so it takes **milliseconds**,
   not the ~30-40s a full prompt replay costs (that was the first version of this fix, and it
   showed up in testing as a ~35s dead-audio stall that dropped mic input the whole time; the
   instant-restore version has no such stall). Mic audio queued up during the last few
   milliseconds is discarded (it predates the restored state).
2. **Per-interval health logging:** the `perf:` line includes `text_tokens=` (words spoken in the
   interval), `out_rms=` (output loudness), and `last_text=` (seconds since the model last said
   anything) -- so "model went mute" vs "pipeline broke" is decidable from the log alone.
3. **Client reconnect + server heartbeat** (`useSocket.ts`, `heartbeat=15`): fixes the
   *secondary* failure mode where any transient network drop permanently ended the session
   (there was no reconnect logic at all, plus an aggressive 10s inactivity kill-switch).
   Requires the client built from source (Section 9b) -- NVIDIA's prebuilt `dist.tgz` predates it.

**What this fix genuinely cannot do -- read this before assuming otherwise:**

- **It does not preserve the conversation across a recovery.** The restored state *is* the
  "persona just loaded, nothing said yet" state -- that's the entire point (it's the last point
  before things went wrong). The model's fixed-size attention window is exactly what couldn't
  hold "persona + all the conversation since" in the first place, and that's still true here: a
  window that could hold both would never have gone mute to begin with. Recovery buys you a
  *fresh* window with the persona back in it, not a merged one.
- **The model will typically re-introduce itself / greet again after a recovery.** The restored
  state is, from the model's point of view, indistinguishable from the very first moment the
  conversation started (same tokens, same position) -- it wasn't trained on "system prompt
  reappearing mid-conversation" as a distinct situation, so it reacts to it the same way it did
  the first time: with an opening line. There's no reliable way to suppress that from the
  outside without retraining the model; claiming otherwise would be overselling this fix.
- **A 10s trigger will sometimes fire during a normal conversational pause**, not just a genuine
  "gone silent forever" event (see the `MUTE_RECOVERY_SECS` comment in Section 10). Watch for the
  model interrupting itself mid-thought; that's the signature of a false trigger, and the fix is
  to raise the threshold, not to chase the trigger logic further -- 10s vs. a real stuck-forever
  state genuinely looks the same from the outside until you wait a bit longer.

**The single most effective thing you actually control: shorten the Text Prompt.** Every
character costs attention-window steps *and* the initial connect delay. A tight "role sentence +
Information: facts" prompt (Section 13b) instead of a multi-thousand-character document gives the
conversation several extra minutes before the first recovery is even needed -- fewer recoveries
means fewer re-greetings, which is the only real lever against that side effect.

**For 30-60+ minute sessions:** accept the model architecturally cannot remember more than ~4
minutes of combined prompt + conversation, and that each recovery is a fresh start for the
persona (not the chat). Keep prompts short to push recoveries further apart, and tune
`MUTE_RECOVERY_SECS` against the natural pause lengths you see in your own `perf:` log's
`last_text=` values rather than assuming 10s is right for every deployment.

## 15c. Forensic instrumentation for the ~6:05 determinism (one-reproduction evidence)

Section 15b explains *why* the model goes silent (the fixed context window). This section is
about *proving exactly what changes at the moment it happens*, from one reproduction, via
structured logging -- not more theory.

**What gets written**, all under `DIAG_DIR` (Section 10):
- `personaplex_diag.jsonl` — one JSON line every `DIAG_INTERVAL_SECS` (default 5s) covering
  timing/offset/wrap position, LM state, audio pipeline counters, connection/task health, and
  performance (GPU/CPU, encode/step/decode time breakdown) -- plus a line for every discrete
  **event**: a `RingKVCache` wrap, every 250-step offset milestone, the offset reaching/exceeding
  3000, output audio going near-silent, 5+ seconds with no text, a mute-recovery trigger, a
  WebSocket disconnect, a task exiting, or any exception.
- `snapshot_<conn_id>_<label>.json` — a full, tensor-free dump of the model's per-layer KV-cache
  state, taken automatically at session start, at 2:00/3:00/4:00/5:00/5:30/5:45/6:00 elapsed, the
  instant 5+ seconds of silence is detected, and 30s after that.
- The moment silence is detected, the server **automatically diffs** the silence-onset snapshot
  against the session-start snapshot and writes the full field-by-field difference as an
  `auto_diff` record in the JSONL stream -- this is the first thing to look at.

**How to reproduce:** run Section 10 with `MUTE_RECOVERY_SECS = 0` (already set above) so the
natural failure isn't auto-corrected before it's fully captured, then have one normal conversation
past the point where it usually goes silent. No other change to how you use the app is needed.

**Overhead:** the periodic/event logging and per-step offset/wrap bookkeeping is arithmetic only
(no extra GPU syncs in the hot path) and negligible; snapshots do real GPU reads but only fire a
handful of times per session. `DIAG_PROBS` is the one opt-in exception -- it adds a small
per-step cost (an extra softmax + logits clone) to capture the model's own token-sampling
probabilities, off by default.

**After reproducing, see the chat response for exactly which log lines to check first.**

## 15d. Follow-up reproduction: mechanism-level evidence (logits, hidden-state norms, attention)

Section 15c's capture proved *when* the collapse happens (offset ≈ 2×capacity) and that the
pipeline stays healthy throughout. It did not explain *why* the model's own output changes at
that point. Three more flags target that, each opt-in because each adds real (if modest)
per-step cost -- set `MUTE_RECOVERY_SECS = 0` again for this run too, and set
`DIAG_DUMP_NEAR` to the `offset` value where the *previous* run's silence began (check
`personaplex_diag.jsonl` for the last `silence_onset` event, or just use `6000` as a starting
guess if you don't have a prior run yet):

```python
DIAG_PROBS = True              # required for DIAG_DUMP_NEAR to capture anything
DIAG_DUMP_NEAR = 6000           # state.offset to center the deep dump on
DIAG_DUMP_RADIUS = 100          # steps on either side to also capture
DIAG_TRACK_NORMS = True         # track the shared hidden-state's L2 norm every step
```

add to the `cmd` list in Section 10:
```python
if DIAG_DUMP_NEAR > 0:
    cmd += ["--diag-dump-near", str(DIAG_DUMP_NEAR), "--diag-dump-radius", str(DIAG_DUMP_RADIUS)]
if DIAG_TRACK_NORMS:
    cmd += ["--diag-track-norms"]
```

This writes `personaplex_deep_dump.jsonl` under `DIAG_DIR`: one record per generation step in
the target window, with the top-20 candidate tokens and probabilities, entropy, and PAD/EPAD
probability -- so a sudden jump in PAD probability (vs. a gradual ramp) is directly visible
without guessing. Hidden-state norms (with `DIAG_TRACK_NORMS`) show up in every snapshot from
that point on (`snapshot_*.json` → `lm_gen.last_hidden_state_norm`), and in the automatic
before/after diff.

**Attention weights (optional, more invasive):** `F.scaled_dot_product_attention` doesn't expose
attention weights by design (that opacity is what makes it fast). To inspect them, the main
transformer must run in eager mode instead of as a captured CUDA graph — start the server with
`NO_CUDA_GRAPH=1` in the environment (already-supported by this codebase, see
`moshi/moshi/utils/compile.py`), and before the first request, enable capture:
```python
import moshi.modules.transformer as T
T.DIAG_CAPTURE_ATTENTION = True
```
Then `T.DIAG_ATTENTION_LOG` accumulates a per-layer, per-step summary (effective attention span,
entropy, max attention weight) for as long as capture is on. This trades throughput for
introspection -- expect the conversation to feel noticeably less real-time-responsive during
this specific run, which is an acceptable one-time cost for this diagnostic, not something to
leave on normally.

## 15e. `--investigate-collapse`: automatic runtime investigation + A/B experiments

Sections 15c/15d require manually picking flags and a target offset. This mode automates
the whole thing: it watches `state.offset` itself, turns on dense capture only for the
window that matters, and writes an evidence-based `investigation_report.md` when it's done
-- no offset guessing required.

```python
INVESTIGATE_COLLAPSE = True
INVESTIGATE_START = 5500       # dense capture begins here
INVESTIGATE_END = 6500         # dense capture ends here, report is generated
INVESTIGATE_DIR = os.path.join(WORKSPACE, "runtime_logs")
EXPERIMENT = "A"               # A=baseline, B=fix RingKVCache bug, C=RoPE rebase,
                                # D=greedy decoding, E=periodic persona refresh, F=one-time reset
```
add to the `cmd` list in Section 10:
```python
if INVESTIGATE_COLLAPSE:
    cmd += ["--investigate-collapse",
            "--investigate-start", str(INVESTIGATE_START),
            "--investigate-end", str(INVESTIGATE_END),
            "--investigate-dir", INVESTIGATE_DIR,
            "--experiment", EXPERIMENT]
```

**What this produces**, under `INVESTIGATE_DIR`:
- `logits.jsonl` -- top-50 tokens/probs, entropy, PAD/EPAD probability, argmax vs. sampled, every step in the window.
- `hidden_state.jsonl` -- shared hidden-state norm, cosine similarity + L2 distance to the previous step, running average.
- `attention.jsonl` -- per-layer entropy, max/avg attention weight, active vs. ignored KV entries (this is the one that requires disabling CUDA graphs -- handled automatically: `--investigate-collapse` forces `NO_CUDA_GRAPH=1` for the whole run, so expect reduced throughput).
- `ringkvcache.jsonl` -- per-layer write pointer/oldest/newest/wrap count, plus a live boundary-slot correctness check every step.
- `rope.jsonl` -- absolute vs. rebased position fed to RoPE, and the resulting angle range.
- `pad_events.jsonl` -- fires once each time PAD probability first crosses 10/20/50/80/95%, with the last 50 tokens and current hidden-state norm attached.
- `before_silence_snapshot.pt` / `after_silence_snapshot.pt` -- full per-layer K/V tensors (not just metadata) at window entry/exit, for offline comparison.
- `investigation_report_<conn>_<experiment>.md` + `experiment_summary_<conn>_<experiment>.json` -- auto-generated from what THIS run actually measured (Proven facts / Rejected hypotheses / Supported hypotheses / Remaining unknowns / Suggested next experiment). Named per-connection-per-experiment (not the literal `investigation_report.md`) so running the A-F comparison doesn't have each run silently overwrite the last one's report -- the summary JSON is meant to be diffed programmatically across experiment runs.

**The six experiments** (`--experiment {A..F}`), each a real, verified code change, not a mock:
- **A** -- baseline, nothing modified.
- **B** -- fixes the proven `RingKVCache` boundary off-by-one (`delta < 0` instead of `delta <= 0`).
- **C** -- feeds RoPE `offset % context` instead of the raw absolute offset, for the rotation angle *only* (the causal mask and cache indexing still use the true absolute offset) -- isolates whether unbounded RoPE positions contribute.
- **D** -- forces greedy decoding (`argmax`, no sampling) for the whole session.
- **E** -- re-feeds the text-prompt tokens every `--experiment-e-refresh-interval` steps (default 500), without any state reset -- the most speculative option, testing whether periodic reinforcement prevents the collapse from ever being reached.
- **F** -- one-time hidden-state reset at `--experiment-f-reset-at` (default offset 5000), reusing the exact same instant-restore mechanism as mute recovery, without reconnecting.

**To actually answer "why", not just "when":** run baseline (A) once, then run B, C, D one at a time (each is a separate server run/reproduction), and compare each `experiment_summary_*.json` against the baseline's -- specifically, whether `pad_prob_range`/`pad_thresholds_crossed` still show the same collapse signature, shift to a different offset, or disappear. That comparison is the actual answer; no single run alone proves causation.

**Caveats, stated plainly:** `NO_CUDA_GRAPH=1` runs the model in eager mode for the whole session, which is slower and (in principle, though not expected to matter here) could execute slightly different fused kernels than normal graphed operation -- if experiment A itself doesn't reproduce the collapse at the same offset as prior graphed runs, that's worth noting rather than assuming away. Each `--investigate-collapse` run only characterizes one conversation with one persona/voice prompt; it is not a claim about every possible configuration.

## 16. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `401`/`403` downloading model assets | License not accepted, or `HF_TOKEN` doesn't belong to the account that accepted it | Re-check Section 7: accept the license at the model page with the **same** account whose token you pasted |
| `no kernel image is available for execution on the device` | Torch build doesn't have Blackwell (`sm_120`) kernels | Re-run the `cu130` reinstall cell in Section 5, then re-run Section 6's smoke test |
| `ImportError` / build errors mentioning `opus` while installing `sphn` | `libopus-dev` missing | Re-run Section 3, then re-run Section 5 |
| Server process exits immediately, log shows a CUDA OOM | Not enough free VRAM (e.g. another process holds the GPU) | Check `nvidia-smi`; consider `USE_CPU_OFFLOAD = True` (Section 2) and re-run Sections 10‑11 |
| Browser blocks microphone / `getUserMedia` fails | Page wasn't loaded over a secure context | Use the RunPod **proxy** URL from Section 11 (HTTPS at the edge), not a plain `http://<pod-ip>:8998` URL |
| Can't reach the URL from outside the pod at all | Port not exposed | In the RunPod console, add `SERVER_PORT` as an HTTP Service on the pod's Connect page |
| First launch seems to hang for several minutes | Normal — first run downloads multi-GB weights and extracts `voices.tgz`/`dist.tgz` | Watch the log tail printed by Section 10; increase `TIMEOUT_S` if your network is slow |
| `mkcert` warnings in the log | Only relevant when `USE_APP_TLS = True`; `moshi/moshi/utils/connection.py` falls back to plain HTTP automatically if `mkcert` can't be installed | Safe to ignore in default (proxy) mode |
| Model goes permanently silent after ~5-6 minutes while the connection stays green and the server `perf:` log stays healthy (`last_text=` keeps climbing, `text_tokens=0`) | The model's fixed 3000-step (~4 min) attention window has fully rotated past the persona prompt; generation degenerates into endless PAD/silence — see **Section 15b** | Mute recovery (`MUTE_RECOVERY_SECS` in Section 10, default 10) restores the persona in milliseconds automatically; shorten the Text Prompt to push the first recovery further out |
| Model briefly restarts/re-greets mid-conversation even though things seemed fine | A mute-recovery cycle fired (log: `model was silent for >Ns -- restored persona`) — either a real mute, or a normal pause longer than `MUTE_RECOVERY_SECS` | Check the log line's timing; if it's firing during ordinary pauses, raise `MUTE_RECOVERY_SECS` (Section 10) |
| Long pause (~40s) only on the very first connect, not on later recoveries | Every Text Prompt token is one full model step loaded before the conversation starts — this only happens once per connection, mute recovery no longer repeats it | Shorten the Text Prompt (Section 13b) to cut this initial pause too |

### Recap: what genuinely cannot be automated by this notebook
1. Clicking "Agree and access repository" on the gated HF model page.
2. Issuing the HF access token for that account.
3. Exposing `SERVER_PORT` as an HTTP Service in the RunPod console.
4. The browser's microphone-permission prompt.

## 17. Stop the server (run only when you want to shut it down)

This is a management utility cell, **not** part of the linear startup flow — running it will terminate the
backend you just verified above. Run it deliberately when you're done with the session, not as part of a
top-to-bottom "Run All".

In [ ]:
# Intentionally NOT meant to run automatically as part of the startup sequence above.
try:
    server_proc.terminate()
    server_proc.wait(timeout=15)
    print(f"Server process {server_proc.pid} stopped.")
except NameError:
    print("No server_proc in scope -- nothing to stop.")
except subprocess.TimeoutExpired:
    server_proc.kill()
    print(f"Server process {server_proc.pid} killed after not stopping gracefully.")